In [0]:
#Imports
from pyspark.sql.functions import from_json,col,regexp_replace, trim,explode
from pyspark.sql.functions import split
from pyspark.sql.types import ArrayType, StructType, StructField, StringType

#Loading Table  

In [0]:
Games=spark.table("steam_data.default.steam_games")

Review=spark.table("steam_data.default.review")

number of rows and columns

In [0]:
def table_size(df, name):
    data = [(name, df.count(), len(df.columns))]
    spark.createDataFrame(data, ["Table", "Rows", "Cols"]).show()
table_size(Games, "Games")
table_size(Review, "Review")

#Drop Duplicates

In [0]:
Games=Games.dropDuplicates()
Review=Review.dropDuplicates()

print("AFTER DROPPING DUPLICATES")

table_size(Games,"Games")
table_size(Review,"Reviews")




**DUPLICATES NOT FOUND AS THE SIZE IS UNCHANGED AFTER DROPPING DUPLICATES**

#Games Data

Counting the number of null in each columns

In [0]:

null_count={}
for colu in Games.columns:   
    number_of_null = Games.filter(col(colu).isNull()).count()
    null_count[colu]=number_of_null


In [0]:

null_count_df = spark.createDataFrame(null_count.items(), ["column", "null_count"])
null_count_df.show()


In [0]:
Games.columns

As there are 42 columns , I am keeping only the ones which are useful for the analysis

In [0]:
columns_to_keep=['app_id','name','price','price_status','estimated_owners','genres','categories','positive','negative','recommendations','peak_ccu','metacritic_score','user_score','average_playtime_forever','windows','mac','linux','supported_Languages']
Games=Games.select(columns_to_keep)
Games=Games.withColumnRenamed("supported_Languages","supported_languages")
Games=Games.withColumn("price",Games["price"].cast("double"))

#Modified Table

In [0]:
Games.describe().show()

display(Games)

We can see the estimated_owners column in given as a-b or a..b, we can split it into a,b and store (a+b)/2. As there are extra spaces we will use trim to remove them

In [0]:
Games = Games.withColumn(
    "estimated_owners",
    split(col("estimated_owners"), r"\.\.|\s*-\s*")
)

In [0]:
Games = Games.withColumn(
    "estimated_owners",
    (
        regexp_replace(trim(col("estimated_owners").getItem(0)), ",", "").cast("double")
        +
        regexp_replace(trim(col("estimated_owners").getItem(1)), ",", "").cast("double")
    ) / 2
)

In [0]:
Games.select("estimated_owners").show()

Store this as a new table

In [0]:
Games.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("steam_data.default.silver_games")

More processing on the new table

In [0]:
steam_data.default.steam_gamessilver_games=spark.table("steam_data.default.silver_games")
silver_games.describe().show()
display(silver_games)

##Fixing the Price column

In [0]:
from pyspark.sql import functions as F

before = silver_games.count()

silver_games = silver_games.filter(F.col("price").isNull() | (F.col("price") <= 300))

after = silver_games.count()
print(f"Dropped {before - after} rows with price > 300")

#Handling Genre and Category

In [0]:
genre_schema = ArrayType(
    StructType([
        StructField("id", StringType(), True),
        StructField("description", StringType(), True)
    ])
)

silver_games = silver_games.withColumn(
    "genres_parsed",
    from_json(col("genres"), genre_schema)
)

In [0]:
game_genre_df = (
    silver_games
    .select(
        "app_id",
        explode(col("genres_parsed")).alias("genres")
    )
    .select(
        "app_id",
        col("genres.id").alias("genre_id")
    )
)

In [0]:
game_genre_df.show()

In [0]:
genres_df = (
    silver_games
    .select(explode(col("genres_parsed")).alias("genres"))
    .select(
        col("genres.id").alias("genre_id"),
        col("genres.description").alias("description")
    )
    .dropDuplicates(["genre_id"])
)

In [0]:
genres_df.show()

In [0]:
game_genre_df.write\
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("steam_data.default.game_genres")

In [0]:
genres_df.write\
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("steam_data.default.genres")

Genres table and Genres-Games table has been created

SOME VISUALIZATION  

In [0]:
%sql
SELECT 
    gen.description,
    COUNT(*) AS game_count
FROM steam_data.default.genres gen
JOIN steam_data.default.game_genres gg
    ON gen.genre_id = gg.genre_id
GROUP BY gen.description
ORDER BY game_count DESC;

Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

#Removing Genre Col from Silver table

In [0]:
%sql
ALTER TABLE steam_data.default.silver_games
SET TBLPROPERTIES ('delta.columnMapping.mode' = 'name');

In [0]:
%sql
ALTER TABLE steam_data.default.silver_games
DROP COLUMN genres;

In [0]:
silver_games=spark.table("steam_data.default.silver_games")
display(silver_games)

Category

In [0]:
category_schema = ArrayType(
    StructType([
        StructField("id", StringType(), True),
        StructField("description", StringType(), True)
    ])
)

In [0]:
category_df = (
    silver_games
    .select(
        "app_id",
        explode(col("category_parsed")).alias("categories")
    )
    .select(
        "app_id",
        col("genres.id").alias("cateogry_id")
    )
)

In [0]:
category_schema = ArrayType(
    StructType([
        StructField("id", StringType(), True),
        StructField("description", StringType(), True)
    ])
)

silver_games = silver_games.withColumn(
    "category_parsed",
    from_json(col("categories"), category_schema)
)



In [0]:


# Lookup table: one row per category
categories_df = (
    silver_games
    .select(explode(col("category_parsed")).alias("category"))
    .select(
        col("category.id").alias("category_id"),
        col("category.description").alias("description")
    )
    .dropDuplicates(["category_id"])
)

In [0]:
categories_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("steam_data.default.categories")

In [0]:
game_categories_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("steam_data.default.game_categories")

In [0]:
%sql
DROP TABLE IF EXISTS steam_data.default.game_category;

In [0]:
%sql
SELECT c.description, COUNT(*) AS game_count
FROM steam_data.default.categories c
JOIN steam_data.default.game_categories gc
  ON c.category_id = gc.category_id
GROUP BY c.description
ORDER BY game_count DESC;

Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

#QUESTIONS TO BE ANSWERED   

##Which price bands earn the best reviews and the most owners?

In [0]:
%sql
SELECT app_id, name, price, price_status, positive + negative AS total_reviews
FROM steam_data.default.silver_games

ORDER BY total_reviews DESC;

In [0]:
%sql
CREATE OR REPLACE TABLE steam_data.default.silver_games AS
SELECT * EXCEPT (categories)
FROM steam_data.default.silver_games;

In [0]:
%sql
CREATE OR REPLACE TABLE steam_data.default.silver_games_filtered AS
SELECT *
FROM steam_data.default.silver_games
WHERE price < 300;

In [0]:
%sql
CREATE OR REPLACE TABLE steam_data.default.gold_games AS
WITH cat_flags AS (
    SELECT gc.app_id,
           MAX(CASE WHEN c.description LIKE '%Multi-player%' THEN 1 ELSE 0 END) AS is_multiplayer,
           MAX(CASE WHEN c.description LIKE '%Co-op%'        THEN 1 ELSE 0 END) AS is_coop
    FROM steam_data.default.game_categories gc
    JOIN steam_data.default.categories c ON gc.category_id = c.category_id
    GROUP BY gc.app_id
),
genre_counts AS (
    SELECT app_id, COUNT(*) AS genre_count
    FROM steam_data.default.game_genres
    GROUP BY app_id
)
SELECT
    g.app_id,
    g.name,
    g.price,
    g.price = 0 AS is_free,
    CASE
        WHEN g.price IS NULL THEN 'Unknown'
        WHEN g.price = 0     THEN '1. Free'
        WHEN g.price < 10    THEN '2. Under 10'
        WHEN g.price < 20    THEN '3. 10 to 20'
        WHEN g.price < 40    THEN '4. 20 to 40'
        ELSE                      '5. 40+'
    END AS price_band,
    g.estimated_owners,
    g.positive,
    g.negative,
    g.positive + g.negative AS total_reviews,
    CASE WHEN g.positive + g.negative > 0
         THEN g.positive / (g.positive + g.negative) END AS positive_ratio,
    g.peak_ccu,
    g.average_playtime_forever,
    CASE WHEN g.metacritic_score > 0 THEN g.metacritic_score END AS metacritic_score,
    g.windows, g.mac, g.linux,
    COALESCE(cf.is_multiplayer, 0) AS is_multiplayer,
    COALESCE(cf.is_coop, 0)        AS is_coop,
    COALESCE(gc.genre_count, 0)    AS genre_count
FROM steam_data.default.silver_games g
LEFT JOIN cat_flags cf     ON g.app_id = cf.app_id
LEFT JOIN genre_counts gc  ON g.app_id = gc.app_id;